### Data load and sanitization

Loads Nsight Systems SQLite exports (`*.sqlite`) from a directory, merges every profile into **one** table, and tags each row with the **GPU index** parsed from the filename (`profile_<rank>_<gpu>.sqlite`).

Events are stacked with an `event_kind` column: `kernel`, `memcpy`, `sync`, `nvtx`. NVTX rows carry `nvtx_*` columns (resolved text and enum names); other kinds have those fields as NaN. Timestamps `start` / `end` are in nanoseconds (trace time).

In [ ]:
from __future__ import annotations

import re
import sqlite3
from pathlib import Path

import pandas as pd

# Directory containing profile_*.sqlite (adjust if you copy traces elsewhere)
PROFILE_DIR = Path("/path/to/profiles/clean_profiles_hero")

assert PROFILE_DIR.is_dir(), f"Missing profile directory: {PROFILE_DIR}"

In [ ]:
def list_profile_sqlites(root: Path) -> list[Path]:
    paths = sorted(root.glob("profile_*.sqlite"))
    if not paths:
        raise FileNotFoundError(f"No profile_*.sqlite under {root}")
    return paths


def parse_gpu_index(path: Path) -> int:
    """Filename pattern: profile_<rank>_<gpu_index>.sqlite"""
    m = re.match(r"^profile_\d+_(\d+)\.sqlite$", path.name)
    if not m:
        raise ValueError(f"Cannot parse GPU index from {path.name!r}")
    return int(m.group(1))


sqlite_paths = list_profile_sqlites(PROFILE_DIR)
sqlite_paths

[PosixPath('/path/to/profiles/clean_profiles_hero/profile_0_0.sqlite'),
 PosixPath('/path/to/profiles/clean_profiles_hero/profile_0_1.sqlite'),
 PosixPath('/path/to/profiles/clean_profiles_hero/profile_0_2.sqlite'),
 PosixPath('/path/to/profiles/clean_profiles_hero/profile_0_3.sqlite')]

In [ ]:
SQL_KERNELS = """
SELECT
    k.start,
    k."end",
    k.deviceId,
    k.contextId,
    k.streamId,
    k.correlationId,
    k.globalPid,
    k.gridX, k.gridY, k.gridZ,
    k.blockX, k.blockY, k.blockZ,
    demangled.value AS demangled_name,
    short.value AS short_name
FROM CUPTI_ACTIVITY_KIND_KERNEL AS k
LEFT JOIN StringIds AS demangled ON k.demangledName = demangled.id
LEFT JOIN StringIds AS short ON k.shortName = short.id
"""

SQL_MEMCPY = """
SELECT
    m.start,
    m."end",
    m.deviceId,
    m.contextId,
    m.streamId,
    m.correlationId,
    m.globalPid,
    m.bytes,
    m.copyKind,
    m.srcKind,
    m.dstKind,
    m.srcDeviceId,
    m.dstDeviceId
FROM CUPTI_ACTIVITY_KIND_MEMCPY AS m
"""

SQL_SYNC = """
SELECT
    s.start,
    s."end",
    s.deviceId,
    s.contextId,
    s.streamId,
    s.correlationId,
    s.globalPid,
    s.syncType,
    s.eventId,
    s.eventSyncId
FROM CUPTI_ACTIVITY_KIND_SYNCHRONIZATION AS s
"""

SQL_NVTX = """
SELECT
    n.start,
    n."end",
    n.eventType AS nvtx_event_type_id,
    et.name AS nvtx_event_type,
    n.rangeId AS nvtx_range_id,
    n.category AS nvtx_category,
    n.color AS nvtx_color,
    COALESCE(n.text, sid.value, '') AS nvtx_text,
    n.domainId AS nvtx_domain_id,
    n.globalTid AS nvtx_global_tid
FROM NVTX_EVENTS AS n
LEFT JOIN StringIds AS sid ON n.textId = sid.id
LEFT JOIN ENUM_NSYS_EVENT_TYPE AS et ON n.eventType = et.id
"""

In [ ]:
def connect_sqlite_ro(path: Path) -> sqlite3.Connection:
    return sqlite3.connect(f"file:{path.resolve()}?mode=ro", uri=True)


def load_profile_events(path: Path, gpu_index: int) -> pd.DataFrame:
    """Read kernel/memcpy/sync rows from one SQLite export; tag with gpu and source file."""
    frames: list[pd.DataFrame] = []

    with connect_sqlite_ro(path) as con:
        kernels = pd.read_sql_query(SQL_KERNELS, con)
        memcpy = pd.read_sql_query(SQL_MEMCPY, con)
        sync = pd.read_sql_query(SQL_SYNC, con)
        nvtx = pd.read_sql_query(SQL_NVTX, con)

    kernels["event_kind"] = "kernel"
    memcpy["event_kind"] = "memcpy"
    sync["event_kind"] = "sync"
    nvtx["event_kind"] = "nvtx"

    # For a unified label column across event kinds.
    nvtx["short_name"] = nvtx["nvtx_text"].fillna("")

    frames.extend([kernels, memcpy, sync, nvtx])

    out = pd.concat(frames, ignore_index=True, sort=False)
    out.insert(0, "gpu_index", gpu_index)
    out["profile_file"] = path.name
    return out


parts: list[pd.DataFrame] = []
for p in sqlite_paths:
    gi = parse_gpu_index(p)
    parts.append(load_profile_events(p, gi))

df_raw = pd.concat(parts, ignore_index=True)
df_raw.shape

(11510581, 35)

In [ ]:
### Select time window

# Apply a global time window directly on the raw dataframe.
# Window is defined in seconds relative to the common GPU start.
per_gpu_bounds = (
    df_raw.groupby("gpu_index", observed=False)
    .agg(gpu_start=("start", "min"), gpu_stop=("end", "max"))
    .reset_index()
)

common_start_ns = int(per_gpu_bounds["gpu_start"].max())
common_stop_ns = int(per_gpu_bounds["gpu_stop"].min())
assert common_start_ns <= common_stop_ns, "Invalid common window: start > stop"

# Set explicit range in seconds (relative to common_start_ns).
WINDOW_START_SEC = 86.0
WINDOW_STOP_SEC = 100.0

requested_start_ns = common_start_ns + int(WINDOW_START_SEC * 1e9)
requested_stop_ns = common_start_ns + int(WINDOW_STOP_SEC * 1e9)

# Clamp to available common range.
global_start_time = max(common_start_ns, requested_start_ns)
global_stop_time = min(common_stop_ns, requested_stop_ns)
assert global_start_time <= global_stop_time, (
    "Requested time window is outside available trace overlap"
)

in_window = (df_raw["start"] <= global_stop_time) & (df_raw["end"] >= global_start_time)
df_windowed_raw = df_raw.loc[in_window].copy()

df_windowed_raw["start"] = df_windowed_raw["start"].clip(lower=global_start_time)
df_windowed_raw["end"] = df_windowed_raw["end"].clip(upper=global_stop_time)
df_windowed_raw["duration_ns"] = df_windowed_raw["end"] - df_windowed_raw["start"]

print(
    f"Requested window: [{WINDOW_START_SEC:.3f}, {WINDOW_STOP_SEC:.3f}] s from common start"
)
print(
    f"Applied window:   [{(global_start_time - common_start_ns) / 1e9:.3f}, {(global_stop_time - common_start_ns) / 1e9:.3f}] s"
)
print(f"Rows before: {len(df_raw):,} | after windowing: {len(df_windowed_raw):,}")

(
    df_windowed_raw.groupby("gpu_index", observed=False)
    .agg(
        rows=("event_kind", "size"), min_start=("start", "min"), max_end=("end", "max")
    )
    .reset_index()
)

Requested window: [86.000, 100.000] s from common start
Applied window:   [86.000, 100.000] s
Rows before: 11,510,581 | after windowing: 266,214


,gpu_index,rows,min_start,max_end
0,0,65945,86065336000,1.000653e+11
1,1,66385,86065336000,1.000653e+11
2,2,67618,86065336000,1.000653e+11
3,3,66266,86065336000,1.000653e+11


### Transforming dataset

In [ ]:
# Pivot start/end into a per-event timeline (one row per timestamped event)
# Keep kernels only because we want kernel names.
# Use the windowed raw dataframe built earlier.
df_select = df_windowed_raw[["gpu_index", "short_name", "start", "end", "event_kind"]]
df_kernel = df_select.loc[df_select["event_kind"] == "kernel"].copy()

df_kernel["kernel_name"] = df_kernel["short_name"].fillna("<unknown>")

start_rows = (
    df_kernel[["gpu_index", "kernel_name", "start", "event_kind"]]
    .rename(columns={"start": "event_time"})
    .assign(event_type="start")
)

end_rows = (
    df_kernel[["gpu_index", "kernel_name", "end", "event_kind"]]
    .rename(columns={"end": "event_time"})
    .assign(event_type="end")
)

df_timeline = pd.concat([start_rows, end_rows], ignore_index=True)
df_timeline = df_timeline.sort_values(
    ["gpu_index", "event_time", "event_type", "kernel_name"], kind="stable"
).reset_index(drop=True)

# Normalize timestamps per GPU by subtracting each GPU's minimum timestamp.
min_ts_per_gpu = df_timeline.groupby("gpu_index", observed=False)[
    "event_time"
].transform("min")
df_timeline["event_time_norm_ns"] = (df_timeline["event_time"] - min_ts_per_gpu).astype(
    "int64"
)

df_timeline[
    ["gpu_index", "event_time", "event_time_norm_ns", "event_type", "kernel_name"]
].head(20)

,gpu_index,event_time,event_time_norm_ns,event_type,kernel_name
0,0,8.606534e+10,0,start,ncclDevKernel_SendRecv
1,0,8.606832e+10,2987379,end,ncclDevKernel_SendRecv
2,0,8.606833e+10,2989203,start,elementwise_kernel
3,0,8.606849e+10,3155219,end,elementwise_kernel
4,0,8.606849e+10,3156723,start,vectorized_elementwise_kernel
5,0,8.606849e+10,3158579,end,vectorized_elementwise_kernel
6,0,8.606850e+10,3159795,start,vectorized_elementwise_kernel
7,0,8.606850e+10,3161011,end,vectorized_elementwise_kernel
8,0,8.606850e+10,3162227,start,vectorized_elementwise_kernel
9,0,8.606850e+10,3163987,end,vectorized_elementwise_kernel


In [ ]:
# Sanity check: each kernel row should have both start and end timestamps
kernel_missing = df_kernel[df_kernel["start"].isna() | df_kernel["end"].isna()].copy()

sanity_summary = (
    df_kernel.assign(
        has_start=df_kernel["start"].notna(),
        has_end=df_kernel["end"].notna(),
    )
    .groupby("gpu_index", observed=False)
    .agg(
        total_kernels=("event_kind", "size"),
        kernels_with_start=("has_start", "sum"),
        kernels_with_end=("has_end", "sum"),
    )
    .assign(
        kernels_missing_any=lambda t: (
            t["total_kernels"]
            - t[["kernels_with_start", "kernels_with_end"]].min(axis=1)
        )
    )
    .reset_index()
)

if kernel_missing.empty:
    print("Sanity check passed: all kernel rows have both start and end timestamps.")
else:
    print(
        f"Sanity check failed: {len(kernel_missing)} kernel rows are missing start or end."
    )

sanity_summary

Sanity check passed: all kernel rows have both start and end timestamps.


,gpu_index,total_kernels,kernels_with_start,kernels_with_end,kernels_missing_any
0,0,22584,22584,22584,0
1,1,22208,22208,22208,0
2,2,22260,22260,22260,0
3,3,22205,22205,22205,0


In [ ]:
# Now assign a label whether it is compute or communication
# Use vectorized string ops and handle missing names safely.
df_timeline["kernel_type"] = (
    df_timeline["kernel_name"]
    .fillna("")
    .str.lower()
    .str.contains("nccl")
    .map({True: "communication", False: "compute"})
)

df_timeline[["gpu_index", "kernel_name", "kernel_type"]].head(200)

,gpu_index,kernel_name,kernel_type
0,0,ncclDevKernel_SendRecv,communication
1,0,ncclDevKernel_SendRecv,communication
2,0,elementwise_kernel,compute
3,0,elementwise_kernel,compute
4,0,vectorized_elementwise_kernel,compute
...,...,...,...
195,0,vectorized_elementwise_kernel,compute
196,0,elementwise_kernel,compute
197,0,elementwise_kernel,compute
198,0,fused_dropout_kernel_vec,compute


In [ ]:
# Sanity check: kernel_type must match kernel_name containing 'nccl' (case-insensitive)
expected_comm = df_timeline["kernel_name"].fillna("").str.lower().str.contains("nccl")
actual_comm = df_timeline["kernel_type"].eq("communication")

label_mismatch = df_timeline.loc[
    expected_comm != actual_comm,
    [
        "gpu_index",
        "kernel_name",
        "event_time",
        "event_type",
        "kernel_type",
    ],
].copy()

if label_mismatch.empty:
    print(
        "Sanity check passed: all 'nccl' kernels are communication and all others are compute."
    )
else:
    print(f"Sanity check failed: found {len(label_mismatch)} mislabeled rows.")

mismatch_summary = (
    df_timeline.assign(
        expected_kernel_type=expected_comm.map(
            {True: "communication", False: "compute"}
        )
    )
    .groupby(["expected_kernel_type", "kernel_type"], observed=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["expected_kernel_type", "kernel_type"], kind="stable")
)

mismatch_summary

Sanity check passed: all 'nccl' kernels are communication and all others are compute.


,expected_kernel_type,kernel_type,rows
0,communication,communication,21488
1,compute,compute,157026


### Overlap analysis

In [ ]:
# Counter timeline: update active compute/communication kernels over time per GPU.
# +1 on start, -1 on end. Counters must never become negative.

df_counter_events = df_timeline[
    [
        "gpu_index",
        "event_time",
        "event_time_norm_ns",
        "kernel_type",
        "event_type",
    ]
].copy()

is_start = df_counter_events["event_type"].eq("start")
is_comm = df_counter_events["kernel_type"].eq("communication")
is_compute = df_counter_events["kernel_type"].eq("compute")

sign = is_start.map({True: 1, False: -1}).astype("int8")

df_counter_events["comm_delta"] = (is_comm.astype("int8") * sign).astype("int8")
df_counter_events["compute_delta"] = (is_compute.astype("int8") * sign).astype("int8")

# Aggregate at identical timestamps first to avoid ordering artifacts when start/end share a timestamp.
df_counter_timeline = (
    df_counter_events.groupby(
        ["gpu_index", "event_time", "event_time_norm_ns"],
        observed=False,
        as_index=False,
    )[["compute_delta", "comm_delta"]]
    .sum()
    .sort_values(["gpu_index", "event_time", "event_time_norm_ns"], kind="stable")
    .reset_index(drop=True)
)

df_counter_timeline["compute_count"] = df_counter_timeline.groupby(
    "gpu_index", observed=False
)["compute_delta"].cumsum()
df_counter_timeline["comm_count"] = df_counter_timeline.groupby(
    "gpu_index", observed=False
)["comm_delta"].cumsum()

# Error out if counters go negative (should never happen).
neg_mask = (df_counter_timeline["compute_count"] < 0) | (
    df_counter_timeline["comm_count"] < 0
)
if neg_mask.any():
    bad = df_counter_timeline.loc[
        neg_mask,
        [
            "gpu_index",
            "event_time",
            "event_time_norm_ns",
            "compute_count",
            "comm_count",
        ],
    ].head(20)
    raise ValueError(
        "Counter underflow detected (compute_count < 0 or comm_count < 0).\n"
        f"First offending rows:\n{bad.to_string(index=False)}"
    )

print("Counter sanity passed: no negative compute/comm counters.")
df_counter_timeline.head(30)

Counter sanity passed: no negative compute/comm counters.


,gpu_index,event_time,event_time_norm_ns,compute_delta,comm_delta,compute_count,comm_count
0,0,8.606534e+10,0,0,1,0,1
1,0,8.606832e+10,2987379,0,-1,0,0
2,0,8.606833e+10,2989203,1,0,1,0
3,0,8.606849e+10,3155219,-1,0,0,0
4,0,8.606849e+10,3156723,1,0,1,0
5,0,8.606849e+10,3158579,-1,0,0,0
6,0,8.606850e+10,3159795,1,0,1,0
7,0,8.606850e+10,3161011,-1,0,0,0
8,0,8.606850e+10,3162227,1,0,1,0
9,0,8.606850e+10,3163987,-1,0,0,0


In [ ]:
# Duration accounting from counter timeline
# - idle: compute_count == 0 and comm_count == 0
# - unoverlapped communication: compute_count == 0 and comm_count > 0

counter_with_next = df_counter_timeline.copy()
counter_with_next["next_event_time"] = counter_with_next.groupby(
    "gpu_index", observed=False
)["event_time"].shift(-1)
counter_with_next["segment_ns"] = (
    counter_with_next["next_event_time"] - counter_with_next["event_time"]
)

valid = counter_with_next["segment_ns"].notna() & (counter_with_next["segment_ns"] >= 0)
idle_mask = (
    (counter_with_next["compute_count"] == 0)
    & (counter_with_next["comm_count"] == 0)
    & valid
)
unoverlapped_comm_mask = (
    (counter_with_next["compute_count"] == 0)
    & (counter_with_next["comm_count"] > 0)
    & valid
)

per_gpu_time = (
    counter_with_next.groupby("gpu_index", observed=False)
    .agg(start_time=("event_time", "min"), end_time=("event_time", "max"))
    .reset_index()
)
per_gpu_time["total_time_ns"] = per_gpu_time["end_time"] - per_gpu_time["start_time"]

idle_ns = (
    counter_with_next.loc[idle_mask]
    .groupby("gpu_index", observed=False)["segment_ns"]
    .sum()
)
unoverlapped_comm_ns = (
    counter_with_next.loc[unoverlapped_comm_mask]
    .groupby("gpu_index", observed=False)["segment_ns"]
    .sum()
)

report_ms = per_gpu_time[["gpu_index", "total_time_ns"]].copy()
report_ms["idle_time_ns"] = report_ms["gpu_index"].map(idle_ns).fillna(0)
report_ms["unoverlapped_comm_time_ns"] = (
    report_ms["gpu_index"].map(unoverlapped_comm_ns).fillna(0)
)
report_ms["compute_time_ns"] = (
    report_ms["total_time_ns"]
    - report_ms["idle_time_ns"]
    - report_ms["unoverlapped_comm_time_ns"]
).clip(lower=0)

report_ms["idle_time_ms"] = report_ms["idle_time_ns"] / 1e6
report_ms["unoverlapped_comm_time_ms"] = report_ms["unoverlapped_comm_time_ns"] / 1e6
report_ms["compute_time_ms"] = report_ms["compute_time_ns"] / 1e6
report_ms["total_time_ms"] = report_ms["total_time_ns"] / 1e6

report_ms["pct_unoverlapped_comm"] = (
    100.0 * report_ms["unoverlapped_comm_time_ns"] / report_ms["total_time_ns"]
)
report_ms["pct_idle"] = 100.0 * report_ms["idle_time_ns"] / report_ms["total_time_ns"]
report_ms["pct_compute"] = (
    100.0 * report_ms["compute_time_ns"] / report_ms["total_time_ns"]
)

report_ms = (
    report_ms[
        [
            "gpu_index",
            "idle_time_ms",
            "unoverlapped_comm_time_ms",
            "compute_time_ms",
            "total_time_ms",
            "pct_unoverlapped_comm",
            "pct_idle",
            "pct_compute",
        ]
    ]
    .sort_values("gpu_index", kind="stable")
    .reset_index(drop=True)
)

overall_totals = {
    "gpu_index": "all",
    "idle_time_ms": report_ms["idle_time_ms"].sum(),
    "unoverlapped_comm_time_ms": report_ms["unoverlapped_comm_time_ms"].sum(),
    "compute_time_ms": report_ms["compute_time_ms"].sum(),
    "total_time_ms": report_ms["total_time_ms"].sum(),
}
overall_totals["pct_unoverlapped_comm"] = (
    100.0
    * overall_totals["unoverlapped_comm_time_ms"]
    / overall_totals["total_time_ms"]
)
overall_totals["pct_idle"] = (
    100.0 * overall_totals["idle_time_ms"] / overall_totals["total_time_ms"]
)
overall_totals["pct_compute"] = (
    100.0 * overall_totals["compute_time_ms"] / overall_totals["total_time_ms"]
)

overall_ms = pd.DataFrame([overall_totals])

print("Per-GPU durations and percentages:")
print(report_ms.to_string(index=False))
print("\nSummed over all GPUs:")
print(overall_ms.to_string(index=False))

report_ms

Per-GPU durations and percentages:
 gpu_index  idle_time_ms  unoverlapped_comm_time_ms  compute_time_ms  total_time_ms  pct_unoverlapped_comm  pct_idle  pct_compute
         0    194.771040                3607.937714     10197.291246        14000.0              25.770984  1.391222    72.837795
         1    112.026273                3535.556739     10352.416988        14000.0              25.253977  0.800188    73.945836
         2    151.483039                3448.645831     10399.871130        14000.0              24.633185  1.082022    74.284794
         3    110.889216                3548.721792     10340.388992        14000.0              25.348013  0.792066    73.859921

Summed over all GPUs:
gpu_index  idle_time_ms  unoverlapped_comm_time_ms  compute_time_ms  total_time_ms  pct_unoverlapped_comm  pct_idle  pct_compute
      all    569.169568               14140.862076     41289.968356        56000.0              25.251539  1.016374    73.732086


,gpu_index,idle_time_ms,unoverlapped_comm_time_ms,compute_time_ms,total_time_ms,pct_unoverlapped_comm,pct_idle,pct_compute
0,0,194.771040,3607.937714,10197.291246,14000.0,25.770984,1.391222,72.837795
1,1,112.026273,3535.556739,10352.416988,14000.0,25.253977,0.800188,73.945836
2,2,151.483039,3448.645831,10399.871130,14000.0,24.633185,1.082022,74.284794
3,3,110.889216,3548.721792,10340.388992,14000.0,25.348013,0.792066,73.859921
